# Netflix Password Sharing Crackdown
*by Austin Zhang, Brynn Zhang, Pranshul Bhatnagar, Sejal Jagtap*

## Causal question
 What is the causal effect of Netflix’s password-sharing crackdown on subscriber growth and revenue?

## Strategy
To isolate the effect more cleanly, we extend the original DiD to a Triple Difference (DDD) design:
**Time difference:** Before vs after crackdown
**Within-Netflix difference:** Regions with crackdown vs not-yet-treated regions
**Across-platform difference:** Netflix vs other streaming platforms (e.g., Disney+)


## Model
$$Y_{r,t,p}=α+β_1(Crackdown_{r,t} × Netflix_p)+γ_r+δ_t+λ_p+ϵ_{r,t,p}$$
Where:
$\gamma_r$​: region fixed effects
$\delta_t$​: time fixed effects
p = platform (Netflix vs Disney+)
$\beta_1$= causal effect

## Data
A panel dataset with 3 dimensions: Region × Time × Platform
**Core Variables:**
- Netflix subscribers by region
- Netflix revenue / ARPU
- Disney+ subscriber data (regional or global proxy)
**Additional constructed variables:**
- Treatment indicator (by region × time)
- Platform indicator (Netflix = 1, others = 0)
- Interaction terms (for triple diff)
**Sample Requirements:**
- Multiple regions (Netflix regions already given)
- Multiple platforms (Netflix + Disney+)
- Pre + post periods 
| Region | Treatment Start               |
| ------ | ----------------------------- |
| LATAM  | 2022 Q3                  |
| EMEA   | 2023 Q1                       |
| UCAN   | 2023 Q2                       |
| APAC   | 2023 Q2                       |



### Step1: DATA CLEANING
1. expected columns: 
| region | quarter | platform | subscribers | revenue | ARPU | subscriber_growth | treated | is_netflix |
| ------ | ------- | -------- | ----------- | ------- | ---- | ----------------- | ------- | ---------- |
2. Update Netflix dataset: standardize time, keep only relevent col, create outcome variables
3. Define Treatment Timing, Create treatment indicator, Add Event-time variable, Add Platform Indicator
4. Update Disney+ dataset: standardize time, create outcome variables, Add platform indicators
5. Combine Datasets
6. Create Triple-Diff Variable

In [1]:
import pandas as pd 
import numpy as np

n_df = pd.read_csv("data/netflix_revenue_updated.csv")
d_df = pd.read_csv("data/disney_streaming_combined.csv")



In [4]:
# Clean column names first
n_df.columns = n_df.columns.str.strip()

# Parse date
n_df["Date"] = pd.to_datetime(n_df["Date"], dayfirst=True)

# Create quarter variable
n_df["quarter"] = n_df["Date"].dt.to_period("Q")

# Keep only the columns needed
netflix = n_df[
    [
        "Date", "quarter",
        "UCAN Streaming Revenue", "EMEA Streaming Revenue", "LATM Streaming Revenue", "APAC Streaming Revenue",
        "UCAN Members", "EMEA  Members", "LATM Members", "APAC Members",
        "UCAN ARPU", "EMEA ARPU", "LATM  ARPU", "APAC  ARPU"
    ]
].copy()

# Rename columns so they are consistent
netflix = netflix.rename(columns={
    "UCAN Streaming Revenue": "UCAN_revenue",
    "EMEA Streaming Revenue": "EMEA_revenue",
    "LATM Streaming Revenue": "LATAM_revenue",
    "APAC Streaming Revenue": "APAC_revenue",
    "UCAN Members": "UCAN_subscribers",
    "EMEA  Members": "EMEA_subscribers",
    "LATM Members": "LATAM_subscribers",
    "APAC Members": "APAC_subscribers",
    "UCAN ARPU": "UCAN_arpu",
    "EMEA ARPU": "EMEA_arpu",
    "LATM  ARPU": "LATAM_arpu",
    "APAC  ARPU": "APAC_arpu"
})


# Reshape wide -> long
netflix_long_parts = []

for region in ["UCAN", "EMEA", "LATAM", "APAC"]:
    temp = netflix[["Date", "quarter"]].copy()
    temp["region"] = region
    temp["platform"] = "Netflix"
    temp["subscribers"] = netflix[f"{region}_subscribers"]
    temp["revenue"] = netflix[f"{region}_revenue"]
    temp["ARPU"] = netflix[f"{region}_arpu"]
    netflix_long_parts.append(temp)

netflix_long = pd.concat(netflix_long_parts, ignore_index=True)

# Sort
netflix_long = netflix_long.sort_values(["region", "quarter"]).reset_index(drop=True)

# Create subscriber growth
netflix_long["subscriber_growth"] = netflix_long.groupby("region")["subscribers"].diff()

netflix_long


,Date,quarter,region,platform,subscribers,revenue,ARPU,subscriber_growth
0,2019-03-31,2019Q1,APAC,Netflix,12141000,319602000,9.37,NaN
1,2019-06-30,2019Q2,APAC,Netflix,12942000,349494000,9.29,801000.0
2,2019-09-30,2019Q3,APAC,Netflix,14485000,382304000,9.29,1543000.0
3,2019-12-31,2019Q4,APAC,Netflix,16233000,418121000,9.07,1748000.0
4,2020-03-31,2020Q1,APAC,Netflix,19835000,483660000,8.94,3602000.0
...,...,...,...,...,...,...,...,...
79,2023-03-31,2023Q1,UCAN,Netflix,74398000,3608645000,16.18,102000.0
80,2023-06-30,2023Q2,UCAN,Netflix,75571000,3599448000,16.00,1173000.0
81,2023-09-30,2023Q3,UCAN,Netflix,77321000,3735133000,16.29,1750000.0
82,2023-12-31,2023Q4,UCAN,Netflix,80128000,3594791000,16.64,2807000.0


In [5]:
# Platform indicator
netflix_long["is_netflix"] = 1

# Treatment rollout timing you chose
rollout = {
    "LATAM": pd.Period("2022Q3", freq="Q"),
    "EMEA": pd.Period("2023Q1", freq="Q"),
    "UCAN": pd.Period("2023Q2", freq="Q"),
    "APAC": pd.Period("2023Q2", freq="Q")
}

# Treated indicator
netflix_long["treated"] = netflix_long.apply(
    lambda x: int(x["quarter"] >= rollout[x["region"]]),
    axis=1
)

# Event time for future event-study analysis
netflix_long["event_time"] = netflix_long.apply(
    lambda x: x["quarter"].ordinal - rollout[x["region"]].ordinal,
    axis=1
)

# Since Netflix is the treated platform, DDD piece will later be treated * is_netflix
netflix_long["DDD"] = netflix_long["treated"] * netflix_long["is_netflix"]

netflix_long

,Date,quarter,region,platform,subscribers,revenue,ARPU,subscriber_growth,is_netflix,treated,event_time,DDD
0,2019-03-31,2019Q1,APAC,Netflix,12141000,319602000,9.37,NaN,1,0,-17,0
1,2019-06-30,2019Q2,APAC,Netflix,12942000,349494000,9.29,801000.0,1,0,-16,0
2,2019-09-30,2019Q3,APAC,Netflix,14485000,382304000,9.29,1543000.0,1,0,-15,0
3,2019-12-31,2019Q4,APAC,Netflix,16233000,418121000,9.07,1748000.0,1,0,-14,0
4,2020-03-31,2020Q1,APAC,Netflix,19835000,483660000,8.94,3602000.0,1,0,-13,0
...,...,...,...,...,...,...,...,...,...,...,...,...
79,2023-03-31,2023Q1,UCAN,Netflix,74398000,3608645000,16.18,102000.0,1,0,-1,0
80,2023-06-30,2023Q2,UCAN,Netflix,75571000,3599448000,16.00,1173000.0,1,1,0,1
81,2023-09-30,2023Q3,UCAN,Netflix,77321000,3735133000,16.29,1750000.0,1,1,1,1
82,2023-12-31,2023Q4,UCAN,Netflix,80128000,3594791000,16.64,2807000.0,1,1,2,1


In [6]:
# Clean column names
d_df.columns = d_df.columns.str.strip()

# Parse date
d_df["Date"] = pd.to_datetime(d_df["Date"])

# Create quarter variable
d_df["quarter"] = d_df["Date"].dt.to_period("Q")

# Rename to match final format
disney = d_df.rename(columns={
    "Revenue_USD_B": "revenue",
    "Subscribers_M": "subscribers",
    "ARPU_USD": "ARPU"
}).copy()

# Add identifiers
disney["platform"] = "Disney+"
disney["region"] = "Global"
disney["is_netflix"] = 0

# Sort and create subscriber growth
disney = disney.sort_values(["region", "quarter"]).reset_index(drop=True)
disney["subscriber_growth"] = disney.groupby("region")["subscribers"].diff()

# Disney+ does not get Netflix treatment
disney["treated"] = 0
disney["event_time"] = np.nan
disney["DDD"] = 0

# Keep same columns as Netflix
disney = disney[
    [
        "Date", "quarter", "region", "platform",
        "subscribers", "revenue", "ARPU",
        "subscriber_growth", "is_netflix",
        "treated", "event_time", "DDD"
    ]
].copy()

#keep same col for netflix
netflix_long = netflix_long[
    [
        "Date", "quarter", "region", "platform",
        "subscribers", "revenue", "ARPU",
        "subscriber_growth", "is_netflix",
        "treated", "event_time", "DDD"
    ]
].copy()

#combine D, N
combined = pd.concat([netflix_long, disney], ignore_index=True)

# Sort final dataset
combined = combined.sort_values(["platform", "region", "quarter"]).reset_index(drop=True)

combined


,Date,quarter,region,platform,subscribers,revenue,ARPU,subscriber_growth,is_netflix,treated,event_time,DDD
0,2019-03-30,2019Q1,Global,Disney+,28,9.000000e-01,10.70,NaN,0,0,NaN,0
1,2019-06-29,2019Q2,Global,Disney+,30,1.000000e+00,11.10,2.0,0,0,NaN,0
2,2019-09-28,2019Q3,Global,Disney+,32,1.100000e+00,11.50,2.0,0,0,NaN,0
3,2019-12-28,2019Q4,Global,Disney+,35,1.200000e+00,11.70,3.0,0,0,NaN,0
4,2020-03-28,2020Q1,Global,Disney+,50,1.600000e+00,12.80,15.0,0,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...
103,2023-03-31,2023Q1,UCAN,Netflix,74398000,3.608645e+09,16.18,102000.0,1,0,-1.0,0
104,2023-06-30,2023Q2,UCAN,Netflix,75571000,3.599448e+09,16.00,1173000.0,1,1,0.0,1
105,2023-09-30,2023Q3,UCAN,Netflix,77321000,3.735133e+09,16.29,1750000.0,1,1,1.0,1
106,2023-12-31,2023Q4,UCAN,Netflix,80128000,3.594791e+09,16.64,2807000.0,1,1,2.0,1


In [7]:
netflix_long.to_csv("netflix_cleaned_long.csv", index=False)
disney.to_csv("disney_cleaned.csv", index=False)
combined.to_csv("streaming_combined_cleaned.csv", index=False)

### final columns illustration 
| Column            | Role                      |
| ----------------- | ------------------------- |
| quarter           | time dimension            |
| region            | unit of comparison        |
| platform          | cross-platform comparison |
| treated           |```1``` → After crackdown in that region; ```0``` → Before crackdown             |
| is_netflix        | treated group             |
| DDD               | **causal effect variable**  ```treated * is_netflix```  |
| subscriber_growth | ```current subscribers - previous quarter subscribers     ```      |
| ARPU              | ```revenue / subscribers``` (for monetization outcome)     |
| event_time        | ```0``` → first treated quarter; ```-1``` → one quarter before; ```+2``` → two quarters after|
